In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
import os
from pathlib import Path

In [ ]:
CACHE_ROOT = "/cs/labs/oabend/tomer.shahaf/hf_cache_root"
research_df_tmp_with_cosines_parquet_path = os.path.join(CACHE_ROOT, "df_sampled_100k_tmp_with_cosines.pqt")
research_high_semantic_conversations_parquet_path = os.path.join(CACHE_ROOT, "research_high_semantic_conversations.pqt")
research_high_semantic_conversations_parquet_output_path = os.path.join(CACHE_ROOT, "research_high_semantic_conversations_parquet_output.pqt")
final_chunks_folder = os.path.join(CACHE_ROOT, "final_chunks")
final_conversations_df_parquet_path = os.path.join(CACHE_ROOT, "final_conversations_df_parquet.pqt")

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook' 

In [ ]:
# conversations_df = pl.read_parquet(research_high_semantic_conversations_parquet_output_path)
# conversations_df.shape

In [ ]:
conversations_df = pl.read_parquet(final_conversations_df_parquet_path)
# conversations_df.write_parquet(final_conversations_df_parquet_path)
conversations_df.shape

In [ ]:
parquet_files = list(Path(final_chunks_folder).glob("*.pqt"))

dfs_list = []
for file in tqdm(parquet_files):
    dfs_list.append(pl.read_parquet(file))

conversations_df = pl.concat(dfs_list)
conversations_df.shape

In [ ]:
# conversations_df = conversations_df[:10_000]

In [ ]:
conversations_df.columns

In [ ]:
px.histogram(conversations_df["count_before_model_semantic_change"])

In [ ]:
px.histogram(conversations_df["model_name"])

In [ ]:
conversations_df["user_prompts"].sample()[0]

In [ ]:
import time

start = time.time()
import pandas as pd
print(f"pandas: {time.time() - start:.2f}s")

start = time.time()
import polars as pl
print(f"polars: {time.time() - start:.2f}s")

start = time.time()
import sklearn
print(f"sklearn: {time.time() - start:.2f}s")

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

In [ ]:
# CONFIG
MAX_FEATURES = 1_000  # from your constant
USE_TFIDF = False  # set True if you prefer TF-IDF weighting instead of raw counts
TOP_TURNS = 4       # how many turns to keep (1..TOP_TURNS)
N_TOP = 10          # how many rising/dropping words to show

In [ ]:
# === 1) Build the per-turn corpus (fast Polars lazy -> collect once) ===
user_prompts_clean_df = (
    conversations_df.lazy()
    .with_row_index(name="conv_id")
    .explode("user_prompts")
    .with_columns(pl.col("user_prompts").cum_count().over("conv_id").alias("turn_index"))
    .filter(pl.col("turn_index") < TOP_TURNS)  # keep only first TOP_TURNS turns
    .group_by("turn_index")
    .agg(pl.col("user_prompts").list.join(" "))  # join per turn in one pass
    .sort("turn_index")
    .collect()
)

In [ ]:
# Convert the joined column to a plain Python list (order matches turn_index)
turns = user_prompts_clean_df["user_prompts"].to_list()

# Defensive: if some turns are missing, fill with empty string
if len(turns) < TOP_TURNS:
    turns += [""] * (TOP_TURNS - len(turns))

# === 2) Vectorize (sparse) ===
Vectorizer = TfidfVectorizer if USE_TFIDF else CountVectorizer
vec = Vectorizer(stop_words="english", max_features=MAX_FEATURES)

X = vec.fit_transform(turns)  # shape: (n_turns, n_features), sparse matrix

feature_names = vec.get_feature_names_out()

# === 3) Row-normalize to probabilities (fast, operates on sparse) ===
# This converts each row to sum=1 (L1 normalization)
X_prob = normalize(X, norm="l1", axis=1, copy=True)

# === 4) Prepare DataFrames for display/analysis ===
# For reasonably small features (<= few thousands) converting to dense for pandas is fine.
# If you expect millions of features, skip the dense conversion and analyze with sparse ops.
probs_arr = X_prob.toarray()               # shape: (n_turns, n_features)
counts_arr = X.toarray()                   # raw counts (or TF-IDF values if TF-IDF used)

# Build pandas DataFrames with turns as columns
turn_cols = [f"Turn {i+1}" for i in range(probs_arr.shape[0])]
df_probs = pd.DataFrame(probs_arr.T, index=feature_names, columns=turn_cols)
df_counts = pd.DataFrame(counts_arr.T, index=feature_names, columns=turn_cols)

# === 5) Compute shifts and top rising/dropping words ===
# Diff = Turn 4 - Turn 1 (if you want to parametrize, replace indices)
diff_col = f"Diff (T{TOP_TURNS} - T1)"
df_probs[diff_col] = df_probs[f"Turn {TOP_TURNS}"] - df_probs["Turn 1"]

drop_df = df_probs.sort_values(diff_col).head(N_TOP).reset_index().rename(columns={"index": "Word"})
rise_df = df_probs.sort_values(diff_col, ascending=False).head(N_TOP).reset_index().rename(columns={"index": "Word"})

# === 6) Build the "rich" table (prob, count, and factor) for a given set of words ===
def make_rich_df(words):
    probs_subset = df_probs.loc[words, turn_cols].reset_index().rename(columns={"index": "Word"})
    counts_subset = df_counts.loc[words, turn_cols].reset_index().rename(columns={"index": "Word"})
    # factor = current_prob / Turn 1 prob ; avoid div by zero by using np.where
    p = df_probs.loc[words, turn_cols]
    turn1 = p["Turn 1"].replace(0, np.nan)  # temporarily set zeros to NaN
    factors = p.div(turn1, axis=0).fillna(np.inf)  # if Turn1 was zero and later prob >0 -> inf
    factors = factors.reset_index().rename(columns={"index": "Word"})
    melt_probs = probs_subset.melt(id_vars="Word", var_name="Turn", value_name="Probability")
    melt_counts = counts_subset.melt(id_vars="Word", var_name="Turn", value_name="Count")
    melt_factor = factors.melt(id_vars="Word", var_name="Turn", value_name="Factor")
    df_rich = melt_probs.merge(melt_counts, on=["Word", "Turn"]).merge(melt_factor, on=["Word", "Turn"])
    # tidy: replace infinities with a large label or value if you prefer
    df_rich["Factor"] = df_rich["Factor"].replace(np.inf, np.nan)  # or keep inf to indicate "new word"
    return df_rich

# Example outputs
print("--- Words that DROP the most (Openers) ---")
print(drop_df)
print("\n--- Words that RISE the most (Follow-ups) ---")
print(rise_df)

# Create rich tables for the top drop and rise lists
words_drop = drop_df["Word"].tolist()
words_rise = rise_df["Word"].tolist()

df_rich_drop = make_rich_df(words_drop)
df_rich_rise = make_rich_df(words_rise)

# Preview
print(df_rich_drop.head())